![Week 1 Day 5](../../assets/banners/w1d5.png)

# Week 1 Day 5 - Solution: How Consistent Is v1?

The Day 5 exercise: run the extraction **twice** and diff the two runs against each other, then extract **without** the model answers and see what changes. Bonus: compare your smart model.

> 💡 Try the exercise yourself first (it's at the end of `week1/day5.ipynb`). Struggling a little is where the learning happens! Then come back and compare.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise.config import DATA_DIR, settings
from gradewise.exam import exam_pack, extract_exam
from gradewise.models import chat_model, make_chat_model
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: The diff helper from Day 5

In [ ]:
from difflib import SequenceMatcher

from rich.table import Table

from gradewise.console import console


def similarity(a: str | None, b: str | None) -> float:
    """0.0 (nothing alike) → 1.0 (identical), ignoring case and extra whitespace."""
    a, b = " ".join((a or "").lower().split()), " ".join((b or "").lower().split())
    return 1.0 if a == b else SequenceMatcher(None, a, b).ratio()


def compare_exams(expected: ExamPaper, actual: ExamPaper, text_threshold: float = 0.9) -> list[dict]:
    """Field-by-field differences between two ExamPapers. An empty list means they match.

    💡 Numbers, IDs and types must match EXACTLY. Free text only has to be similar enough,
       because "copy the text" can still differ by a space or a curly quote.
    """
    rows: list[dict] = []

    def report(where: str, field: str, want, got, kind: str) -> None:
        rows.append({"where": where, "field": field, "expected": want, "actual": got, "kind": kind})

    def text(where: str, field: str, want: str | None, got: str | None, threshold: float = text_threshold) -> None:
        score = similarity(want, got)
        if score < threshold:
            report(where, field, (want or "")[:60], (got or "")[:60], f"text {score:.0%} similar")

    for field in ["title", "subject", "total_marks"]:
        if getattr(expected, field) != getattr(actual, field):
            report("paper", field, getattr(expected, field), getattr(actual, field), "value")

    want_qs = {q.id: q for q in expected.questions}
    got_qs = {q.id: q for q in actual.questions}
    for qid in want_qs.keys() - got_qs.keys():
        report(qid, "question", qid, None, "missing")
    for qid in got_qs.keys() - want_qs.keys():
        report(qid, "question", None, qid, "extra")

    for qid in [q for q in want_qs if q in got_qs]:  # 💡 keep the paper's order
        want, got = want_qs[qid], got_qs[qid]
        for field in ["number", "question_type", "max_marks"]:
            if getattr(want, field) != getattr(got, field):
                report(qid, field, getattr(want, field), getattr(got, field), "value")
        text(qid, "text", want.text, got.text)
        text(qid, "methodology", want.methodology, got.methodology, threshold=0.5)  # 💡 composed, so looser
        if (want.model_answer is None) != (got.model_answer is None):
            report(qid, "model_answer", want.model_answer is not None, got.model_answer is not None, "present?")
        elif want.model_answer:
            text(qid, "model_answer", want.model_answer, got.model_answer)

        want_cs = {c.id: c for c in want.criteria}
        got_cs = {c.id: c for c in got.criteria}
        for cid in want_cs.keys() - got_cs.keys():
            report(cid, "criterion", f"worth {want_cs[cid].marks:g}", None, "missing")
        for cid in got_cs.keys() - want_cs.keys():
            report(cid, "criterion", None, f"worth {got_cs[cid].marks:g}", "extra")
        for cid in [c for c in want_cs if c in got_cs]:
            if want_cs[cid].marks != got_cs[cid].marks:
                report(cid, "marks", want_cs[cid].marks, got_cs[cid].marks, "value")
            text(cid, "description", want_cs[cid].description, got_cs[cid].description, threshold=0.7)
    return rows


def show_diff(rows: list[dict]) -> None:
    """A readable table of compare_exams() rows, or a green tick if there are none."""
    if not rows:
        print("✅ No differences: the two exams match field by field.")
        return
    table = Table(header_style="accent")
    for column in ["where", "field", "expected", "actual", "kind"]:
        table.add_column(column, overflow="fold")
    for row in rows:
        table.add_row(*(str(row[c]) for c in ["where", "field", "expected", "actual", "kind"]))
    console.print(table)
    print(f"{len(rows)} difference(s). The serious ones: missing / extra / value rows.")

In [ ]:
expected = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
pack = exam_pack("cs_midterm")
show_diff(compare_exams(expected, expected))  # 💡 always test your test

## Part 2: Two runs, one model

💡 We create the model once and reuse it, and we read the PDFs once (as text) so both runs see identical input: the only thing that can differ is the model's output.

In [ ]:
from gradewise.exam.parse import read_document

texts = {key: read_document(path) for key, path in pack.items()}  # PDFs → text, once
llm = chat_model(temperature=0)

run_1 = extract_exam(**texts, llm=llm)
run_2 = extract_exam(**texts, llm=llm)

print("Run 1 vs run 2:")
show_diff(compare_exams(run_1, run_2))

**What you'll typically see:** IDs, marks and question types almost never change between runs; that's the structure doing its job. The rows that *do* appear are mostly free text: `methodology` (which the model composes from several sections) and criterion wording. And with OpenAI's reasoning models, `temperature=0` is silently ignored (Day 2), so some variation is expected. This is why week 3 adds validation, retries and **Asha's approval** before an exam is stored.

In [ ]:
print("Run 1 vs the answer key:")
show_diff(compare_exams(expected, run_1))

## Part 3: Without the model answers

In [ ]:
no_answers = {k: v for k, v in texts.items() if k != "model_answers"}
run_3 = extract_exam(**no_answers, llm=llm)

missing = [q.id for q in run_3.questions if q.model_answer is None]
print(f"model_answer is null on {len(missing)} of {len(run_3.questions)} questions")
show_diff(compare_exams(expected, run_3))

Rule 9 of the system prompt says *"`model_answer` is copied from the MODEL ANSWERS document when one is given; otherwise null. Never write your own."* So the right result is `model_answer: null` everywhere, which the diff reports as one `present?` row per question. If your model wrote its own model answers anyway, that's a rule it broke: worth knowing before you trust it with grading!

## Bonus: the smart model

In [ ]:
smart = extract_exam(**texts, llm=make_chat_model(settings.smart_model, temperature=0))
smart_rows = compare_exams(expected, smart)
print(
    f"Smart model ({settings.smart_model}): {len(smart_rows)} differences from the key, "
    f"vs {len(compare_exams(expected, run_1))} for {settings.chat_model}"
)
show_diff(smart_rows)

Is the smart model worth it *here*? Extraction happens **once per exam**, so even a model ten times the price costs pennies. Grading happens once per answer, per student, so that's where model choice really moves the bill. You'll make that trade-off with data in week 5.